In [1]:
#by default the GPTQ uses unsigned implementation
def find_scale_zero(w_min, w_max, bits = 4, symmetric = True):
    qmax = 2**bits - 1
    qmin =0
    
    #scale and zero are used to convert the fp32 to int8 
    
    if symmetric:
        max_abs = torch.maximum(w_min.abs(), w_max.abs())
        max_abs = torch.clamp(max_abs, min = 1e-8)
        scale = max_abs / ((qmax - qmin)/2)
        zero = torch.full_like(scale, (qmin + qmax) /2)
    else:
        w_max = torch.maximum(w_max, w_min)
        range_ = torch.clamp(w_max - w_min, min = 1e-8)
        scale = range_/(qmax - qmin)
        zero = torch.round(qmin - w_min / scale)
        zero = torch.clamp(zero, qmin, qmax)

    return scale, zero


def quantize(w, scale, zero, bits = 4):
    qmax = 2**bits - 1
    qmin = 0
    q = torch.round(w / scale) + zero
    q = torch.clamp(q, qmin, qmax)          # this clamp IS the lossy step
    w_hat = (q - zero) * scale              # dequantize back to real-valued weight
    return w_hat


In [2]:
import math

In [3]:
def group_quantize(W, X, device, bits = 4, block_size = 128, percdamp = 0.01):

    orig_dtype = W.dtype  #dtype of qwen weights
    W = W.float()
    X = X.float()

    
    H = (2 / X.shape[0]) * (X.T.float() @ X.float())
    #caculate importance of cols
    importance = torch.diag(H)
    perm = torch.argsort(importance, descending = True)
    
    H = H[perm] [:, perm] #ts didn;t understand, said it is needed because comes as a pair
    W = W[:, perm]

    
    #dampen to avoid 0 values
    damp = percdamp * torch.mean(torch.diagonal(H)).item()
    H += damp * torch.eye(H.shape[0], H.shape[1], dtype = torch.float32, device=device)

    #hessian inverse and the L value
    L = torch.linalg.cholesky(H).to(device)

    del H
    
    #calculate th inverse of hessian using L
    H_inv = torch.cholesky_inverse(L) #H_inv = L @ L.T
    H_inv_chol = torch.linalg.cholesky(H_inv, upper = True).to(device)

    del L, H_inv

    torch.cuda.empty_cache()
    
    #number of blocks
    num_blocks = math.ceil(W.shape[-1] / block_size)
    
    scales_store = []
    zeros_store = []
    total_cols = W.shape[-1]
    
    for block_idx in range(num_blocks):
        block_start = block_idx * block_size
        block_end = min(block_start + block_size, total_cols)
        B = block_end - block_start
    
        group = W[:, block_start:block_end]
        w_min = group.min(dim=1).values
        w_max = group.max(dim=1).values
        scale, zero = find_scale_zero(w_min, w_max, bits=4, symmetric=False)
    
        scales_store.append(scale.cpu())
        zeros_store.append(zero.cpu())
        
        E = torch.zeros(W.shape[0], B, device = device)
    
        for idx, i in enumerate(range(block_start, block_end)):
            w_i = W[:, i]
            w_i_hat = quantize(w_i, scale, zero)
            e_i = (w_i - w_i_hat) / H_inv_chol[i, i]
            W[:, i] = w_i_hat
            E[:, idx] = e_i
    
            #local update
            if i + 1 < block_end:
                W[:, i+1:block_end] -= e_i.unsqueeze(1) @ H_inv_chol[i, i+1:block_end].unsqueeze(0)
    
        if block_end < total_cols:
            W[:, block_end:] -= E @ H_inv_chol[block_start:block_end, block_end:]

    del E, w_i
    return perm, W, scales_store, zeros_store

In [4]:
# To add a hook to a Pytorch Module, we add a function using pytorch's hook registration method
# A hook allows you to inspect or modify intermediate inputs, outputs or gradients without altering the original module's source code

In [5]:
import torch
from transformers import AutoModelForCausalLM

In [6]:
model_name = "Qwen/Qwen3-4B"

In [7]:
#load the model to cpu
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.float16, #fp16 is supported
    device_map = "cpu"
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [8]:
device_cpu = torch.device("cpu")
device_gpu = torch.device("cuda:0")

In [9]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
from datasets import load_dataset
import random

c4 = load_dataset("allenai/c4", "en", split="train", streaming=True)

def sample_c4_batches(tokenizer, num_batches, batch_size, seq_len, seed=0):
    random.seed(seed)
    c4_iter = iter(c4)
    calib_batches = []

    for _ in range(num_batches):
        batch = []
        while len(batch) < batch_size:
            # skip a random number of rows to get pseudo-random sampling from the stream
            skip = random.randint(0, 1000)
            for _ in range(skip):
                next(c4_iter)
            row = next(c4_iter)
            ids = tokenizer(row["text"], return_tensors="pt").input_ids[0]
            if ids.shape[0] >= seq_len:
                start = random.randint(0, ids.shape[0] - seq_len)
                batch.append(ids[start:start + seq_len])
        calib_batches.append(torch.stack(batch))

    return calib_batches

calib_batches = sample_c4_batches(tokenizer, num_batches=4, batch_size=4, seq_len=512)

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

In [11]:
#take a hook and store the inputs to the model.layers 
layers = model.model.layers

hidden_states = []
layer_inputs = []
layer_kwargs = []
#embedding hook

def embed_hook(mod, inp, out):
    hidden_states.append(out.detach())

#add the hook to the model's embedding
h = model.model.embed_tokens.register_forward_hook(embed_hook)

class Catcher(torch.nn.Module):
    def __init__(self, mod):
        super().__init__()
        self.mod = mod

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.mod, name)

    def forward(self, hidden, **kwargs):
        layer_inputs.append(hidden.detach())
        layer_kwargs.append(kwargs)
        raise RuntimeError("stop")


model.model.embed_tokens.to(device_gpu)
layers[0] = Catcher(layers[0])

for batch in calib_batches:
    try:
        batch = batch.to(device_gpu)
        model(batch)  #for every batch, store the input to layer 0 in lists
    except RuntimeError:
        pass

layers[0] = layers[0].mod
h.remove()

In [12]:
model.model.embed_tokens.to(device_cpu) #free once done

#move the calibration data back to cpu
for batch in calib_batches:
    batch.to(device_cpu)

#empty the cache
torch.cuda.empty_cache()

In [13]:
scales = {}
zeros = {}
perm_order = {}
all_projs = ["q_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"]

In [14]:
for i in range(len(model.model.layers)):
    scales[i] = {projection:[] for projection in all_projs}
    zeros[i] = {projection:[] for projection in all_projs}
    perm_order[i] = {projection:[] for projection in all_projs}

In [ ]:
#the weight matrices of layers are in their original order
#the scales and zeros however are in the permuted order
#perm = [2, 5, 7], means that position 0 in the new permutated matrix contains 2nd column
#so we create a build_g_idx function

In [15]:
def capture_input(module, storage_list):
    def hook(mod, inp, out):
        storage_list.append(inp[0].detach())
    return module.register_forward_hook(hook)
    #returns a handle that can be used to remove hook

#layer_inputs and kwargs are already in gpu from bacth.to(device_gpu)

#to quantize the q, k and v, we need inputs to the matrices
#that is what the above hook is used for
with torch.no_grad():
    for layer_idx, layer in enumerate(layers):
        print(f'{layer.__class__.__name__}_{layer_idx}')
        layer.to(device_gpu) #move to gpu
        #qunatize q, k, v
        cap = {"q" : [], "k" : [], "v" : []}
    
        handles = [
            capture_input(layer.self_attn.q_proj, cap["q"]),
            capture_input(layer.self_attn.k_proj, cap["k"]),
            capture_input(layer.self_attn.v_proj, cap["v"])
        ]
    
        #pass the inputs to the q, k, v
        for x, kw in zip(layer_inputs, layer_kwargs):
            layer(x, **kw)
            
        for handle in handles:
            handle.remove()
    
        #same input for q_proj, k_proj and v_proj
        X_qkv = torch.cat([t.reshape(-1, t.shape[-1]) for t in cap["q"]], dim=0)
        for name in ["q_proj", "k_proj", "v_proj"]:
            W = getattr(layer.self_attn, name).weight.data
            perm, new_weights,scale_store, zero_store  = group_quantize(W, X_qkv, bits = 4, block_size = 128, percdamp = 0.01, device = device_gpu)

            scales[layer_idx][name] = scale_store
            perm_order[layer_idx][name] = perm.cpu()
            zeros[layer_idx][name] = zero_store
    
            #unperute this
            invperm = torch.argsort(perm)
            new_weights = new_weights[:, invperm]
    
            W.copy_(new_weights.to(W.dtype))
            del new_weights, perm, invperm
    
        del cap, X_qkv
            
        #now o_proj
        #make a handle to capture input of o_proj layer
        cap_o = []
        handle = capture_input(layer.self_attn.o_proj, cap_o)
        for x, kw in zip(layer_inputs, layer_kwargs):
            layer (x, **kw)
        handle.remove()

        X_o = torch.cat([t.reshape(-1, t.shape[-1])for t in cap_o], dim=0)
        perm, new_weights, scale_store, zero_store = group_quantize(layer.self_attn.o_proj.weight.data, X_o, bits = 4, block_size = 128, percdamp = 0.01, device = device_gpu)
        # new_weights_list["o_proj"] = new_weights
        # perm_order["o_proj"] = perm_o

        name = "o_proj"
        scales[layer_idx][name] = scale_store
        perm_order[layer_idx][name] = perm.cpu()
        zeros[layer_idx][name] = zero_store
        
        #unpermute o_proj
        invperm = torch.argsort(perm)
        new_weights = new_weights[:, invperm]
        layer.self_attn.o_proj.weight.data.copy_(new_weights.to(layer.self_attn.o_proj.weight.data.dtype))
            
        del new_weights, invperm, perm, cap_o, X_o
    
        #now gate proj and up proj
        cap = {"gate_proj" : [], "up_proj": []}
        handles = [
            capture_input(layer.mlp.gate_proj, cap["gate_proj"]),
            capture_input(layer.mlp.up_proj, cap["up_proj"]),
        ]
    
        for x, kw in zip(layer_inputs, layer_kwargs):
            layer(x, **kw)
            
        for handle in handles:
            handle.remove()
    
        #input to the gate and up layer is same
        X_mlp = torch.cat([t.reshape(-1, t.shape[-1]) for t in cap["gate_proj"]], dim = 0)
        for name in ["gate_proj", "up_proj"]:
            W = getattr(layer.mlp, name).weight.data
            perm, new_weights, scale_store, zero_store = group_quantize(W, X_mlp,  bits = 4, block_size = 128, percdamp = 0.01, device=device_gpu)
            # new_weights_list[name] = new_weights
            # perm_order[name] = perm
            
            scales[layer_idx][name] = scale_store
            perm_order[layer_idx][name] = perm.cpu()
            zeros[layer_idx][name] = zero_store
    
            invperm = torch.argsort(perm)
            new_weights = new_weights[:, invperm]
            W.copy_(new_weights.to(W.dtype))
    
            del new_weights, perm, invperm
        del cap, X_mlp
    
        #down proj 
        #create a handle to capture the input to the down_proj
        cap_down_in = []
        handle = capture_input(layer.mlp.down_proj, cap_down_in)
        new_outputs = []
        
        def out_hook(mod, inp, out):
            new_outputs.append(out.detach())
    
        handle2 = layer.register_forward_hook(out_hook)
    
        for x, kw in zip(layer_inputs, layer_kwargs):
            layer(x, **kw)
            
        handle.remove()
        handle2.remove()
    
        X_down = torch.cat([t.reshape(-1, t.shape[-1]) for t in cap_down_in])
        perm, new_weights, scale_store, zero_store = group_quantize(layer.mlp.down_proj.weight.data, X_down,  bits = 4, block_size = 128, percdamp = 0.01, device = device_gpu)
        # new_weights_list['down_proj'] = new_weights
        # perm_order['down_proj'] = perm

        name = "down_proj"
        scales[layer_idx][name] = scale_store
        perm_order[layer_idx][name] = perm.cpu()
        zeros[layer_idx][name] = zero_store
        
        invperm = torch.argsort(perm)
        new_weights = new_weights[:, invperm]
        layer.mlp.down_proj.weight.data.copy_(new_weights.to(layer.mlp.down_proj.weight.data.dtype))
    
        del perm, new_weights, invperm, cap_down_in, X_down, W
        
        #the input for the next layer
        layer_inputs = [out[0] if isinstance(out, tuple) else out for out in new_outputs]
    
        #bring the layer back to cpu and free that memory from cache
        layer.to(device_cpu)
        torch.cuda.empty_cache()
    
        print(torch.cuda.memory_allocated()/1e9, "GB allocated")
        print(torch.cuda.memory_reserved()/1e9, "GB reservec")
        !nvidia-smi --query-gpu=memory.used,memory.total --format=csv
        print(f'\n')

Qwen3DecoderLayer_0
0.227688448 GB allocated
0.390070272 GB reservec
memory.used [MiB], memory.total [MiB]
543 MiB, 15360 MiB
3 MiB, 15360 MiB


Qwen3DecoderLayer_1
0.372391936 GB allocated
0.746586112 GB reservec
memory.used [MiB], memory.total [MiB]
883 MiB, 15360 MiB
3 MiB, 15360 MiB


Qwen3DecoderLayer_2
0.50765824 GB allocated
0.947912704 GB reservec
memory.used [MiB], memory.total [MiB]
1075 MiB, 15360 MiB
3 MiB, 15360 MiB


Qwen3DecoderLayer_3
0.641875968 GB allocated
1.082130432 GB reservec
memory.used [MiB], memory.total [MiB]
1203 MiB, 15360 MiB
3 MiB, 15360 MiB


Qwen3DecoderLayer_4
0.776093696 GB allocated
1.21634816 GB reservec
memory.used [MiB], memory.total [MiB]
1331 MiB, 15360 MiB
3 MiB, 15360 MiB


Qwen3DecoderLayer_5
0.910311424 GB allocated
1.44703488 GB reservec
memory.used [MiB], memory.total [MiB]
1551 MiB, 15360 MiB
3 MiB, 15360 MiB


Qwen3DecoderLayer_6
1.044529152 GB allocated
1.514143744 GB reservec
memory.used [MiB], memory.total [MiB]
1615 MiB, 15360 MiB
3 

In [16]:
del layer_inputs, new_outputs

In [33]:
for name, obj in list(globals().items()):
    if torch.is_tensor(obj) and obj.is_cuda:
        size_mb = obj.element_size() * obj.nelement() / 1e6
        print(f"{name:20s} {size_mb:8.2f} MB  {tuple(obj.shape)}  {obj.dtype}")

x                       10.49 MB  (4, 512, 2560)  torch.float16
W                       49.81 MB  (2560, 9728)  torch.float16
w_quantized             99.61 MB  (2560, 9728)  torch.float32


In [34]:
import gc

tensors = []
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            tensors.append((obj.element_size() * obj.nelement() / 1e6, tuple(obj.shape), obj.dtype))
    except Exception:
        pass

tensors.sort(reverse=True)
total = sum(t[0] for t in tensors)
print(f"Total tracked: {total/1000:.3f} GB across {len(tensors)} tensors\n")
for size, shape, dtype in tensors[:30]:
    print(f"{size:8.2f} MB  {str(shape):30s}  {dtype}")

Total tracked: 5.035 GB across 311 tensors

   99.61 MB  (2560, 9728)                    torch.float32
   49.81 MB  (2560, 9728)                    torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB  (4, 8, 2048, 128)               torch.float16
   16.78 MB 

In [ ]:
#the cols should only contain 2^bits distinct values
# row = 0
# quantized_vals = new_weights_list["down_proj"][row, 256:384]
# print(quantized_vals)
# print("unique values:", torch.unique(quantized_vals).numel())   # should be ≤ 16 for 4-bit

#down_input_capture = []
# handle = layers[0].mlp.down_proj.register_forward_hook(
#     lambda mod, inp, out: down_input_capture.append(inp[0].detach())
# )

# sample = calib_batches[0].to(device)

# with torch.no_grad():
#     model(sample)
# handle.remove()

#sample_inp = down_input_capture[0].reshape(-1, down_input_capture[0].shape[-1]).float()

# invperm = torch.argsort(perm_order["down_proj"])
# new_weights = new_weights_list["down_proj"][:, invperm]
# new_weights

#print(torch.unique(new_weights_list["down_proj"][0, 128:256].float()).numel())   # should be ≤16 (or a bit more if you added actorder mixing — check per-group as discussed earlier)

# out_orig  = sample_inp @ W_orig_down.T
# out_gptq  = sample_inp @ new_weights.T
# gptq_err  = (out_orig - out_gptq).norm() / out_orig.norm()

# total_cols = W_orig_down.shape[-1]
# block_size = 128
# W_naive = W_orig_down.clone()                    # fix 1

# for blk_start in range(0, total_cols, block_size):
#     blk_end = min(blk_start + block_size, total_cols)
#     grp = W_orig_down[:, blk_start:blk_end]
#     s, z = find_scale_zero(grp.min(dim=1).values, grp.max(dim=1).values, bits=4, symmetric=False)
#     for i in range(blk_start, blk_end):
#         W_naive[:, i] = quantize(W_orig_down[:, i], s, z)

# out_naive = sample_inp @ W_naive.T            # fix 2

# naive_err = (out_orig - out_naive).norm() / out_orig.norm()

# print(f"GPTQ relative output error:  {gptq_err.item():.5f}")
# print(f"Naive relative output error: {naive_err.item():.5f}")
# print(f"Improvement factor: {(naive_err/gptq_err).item():.2f}x")

In [ ]:
ppl_quant = compute_perplexity(model, enc)
print(f'ppl quant', ppl_quant)

In [ ]:
ppl_orig = compute_perplexity(model_orig, enc)
print(f'ppl quant', ppl_orig)

In [19]:
def build_g_idx(perm ,block_size, total_cols):
    g_idx = torch.empty(total_cols, dtype=torch.long, device = perm.device)
    num_blocks = math.ceil(total_cols / block_size)
    for block_idx in range(num_blocks):
        block_start = block_idx * block_size
        block_end = min(block_start + block_size, total_cols)
        orig_cols = perm[block_start : block_end]
        g_idx[orig_cols] = block_idx
    return g_idx.cpu() #g_idx is copied to cpu and g_idx is local, it's gpu reference vanishes


def get_int_codes(W, scale_store, zero_store, g_idx, bits = 4):
    ################ (20), (20), ()
    d_out, d_in = W.shape
    q = torch.empty(W.shape, dtype = torch.uint8, device = W.device)
    qmax = 2**bits - 1
    g_idx = g_idx.to(W.device)  #locally transferred
    #now convert to uint8 for each block of size 128
    for block_idx in range(len(scale_store)):
        cols = (g_idx == block_idx).nonzero(as_tuple=True)[0]
        if len(cols) == 0:
            continue
        scale = scale_store[block_idx].to(W.device)
        zero = zero_store[block_idx].to(W.device)
        w = W[:, cols]
        code = torch.round(w / scale.unsqueeze(1) + zero.unsqueeze(1))
        code = torch.clamp(code, 0, qmax)
        q[:, cols] = code.to(torch.uint8)
    return q.cpu() # is in cpu

In [20]:
def pack_int4(q):
    d_out, d_in = q.shape
    q = q.to(device_gpu)
    if d_in % 2 != 0:
        q  = torch.cat([q, torch.zeros(d_out, 1, dtype=torch.uint8)], dim = 1)
    q_even = q[:, 0::2] #start at 0 and take every 2nd element
    q_odd = q[:, 1::2] #start at 1 and take every 2nd element from 1
    
    #odd element first 4 bits and even the last 4
    packed = (q_odd << 4) | q_even #gpu
    return packed.cpu() #cpu

def unpack_int4(packed, d_in):
    q_even = packed & 0x0F
    q_odd = (packed >> 4) & 0x0F
    
    d_out = packed.shape[0]
    q = torch.empty(d_out, q_even.shape[1] + q_odd.shape[1], dtype = torch.uint8, device = device_gpu)
    q[:, 0::2] = q_even
    q[:, 1::2] = q_odd
    return q[:, :d_in]

In [21]:
def dequant_packed(packed_dict, d_in, bits=4, device = device_gpu):
    
    #packed dict contains the information about 1 quantized weigh, its scales, zeros and g_idx
    
    packed = packed_dict['qweight'].to(device)
    scale_store = packed_dict['scales'] #list
    zero_store = packed_dict['zeros'] #list
    g_idx = packed_dict['g_idx'].to(device)

    q = unpack_int4(packed, d_in).float().to(device)
    
    d_out = q.shape[0]
    W = torch.empty(d_out, d_in, device = device)
    for block_idx in range(len(scale_store)):
        cols = (g_idx == block_idx).nonzero(as_tuple=True)[0]
        if len(cols) == 0:
            continue
        scale = scale_store[block_idx].to(device)
        zero = zero_store[block_idx].to(device)
        W[:, cols] = (q[:, cols] - zero.unsqueeze(1)) * scale.unsqueeze(1)
    return W #in gpu

In [22]:
packed_weights = {}
for layer_idx in range(len(model.model.layers)):
    packed_weights[layer_idx] = {}
    layer = model.model.layers[layer_idx]
    layer.to(device_gpu)
    
    for name in  ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]:
        module = getattr(layer.self_attn, name,None) or getattr(layer.mlp, name)
        W = module.weight.data.float()
        total_cols = W.shape[-1]

        perm = perm_order[layer_idx][name].to(device_gpu)
        scale_store = scales[layer_idx][name]
        zero_store = zeros[layer_idx][name]

        g_idx = build_g_idx(perm, block_size = 128, total_cols = total_cols) #in cpu
        q = get_int_codes(W, scale_store, zero_store, g_idx) #in cpu
        packed = pack_int4(q) #in cpu

        packed_weights[layer_idx][name] = {
            "qweight" : packed, #cpu
            "scales" : scale_store, #cpu
            "zeros" : zero_store, #cpu
            "g_idx" : g_idx #cpu
        }

        del perm
    layer.to(device_cpu)
    torch.cuda.empty_cache()

In [23]:
model.model.layers[0].__class__.__name__

'Qwen3DecoderLayer'

In [24]:
#check difference btwn qunatized and false quantized weight
module_names = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
for layer_idx in range(len(model.model.layers)):

    #get the layer
    layer = model.model.layers[layer_idx]
    print(f'{layer.__class__.__name__}_{layer_idx}\t')
    for name in module_names:
        print(f'{name}\t', end='\t\t')
        #retrieve the weights from false quantized model and packed weights
        module = getattr(layer.self_attn, name,None) or getattr(layer.mlp, name)
        W = module.weight.data.to(device_gpu)

        #retrieve the packed weight
        packed_dict = packed_weights[layer_idx][name]

        w_quantized = dequant_packed(packed_dict, d_in = W.shape[-1], bits=4, device = device_gpu)
    
        print("Max diff:", (w_quantized - W).abs().max().item(), end='\n')   # should be ~0 (tiny fp rounding only)
    print(f'\n')

Qwen3DecoderLayer_0	
q_proj			Max diff: 0.00013023614883422852
k_proj			Max diff: 8.13901424407959e-05
v_proj			Max diff: 5.987286567687988e-05
o_proj			Max diff: 0.0001627206802368164
gate_proj			Max diff: 0.00012108683586120605
up_proj			Max diff: 0.00011640787124633789
down_proj			Max diff: 0.00017565488815307617


Qwen3DecoderLayer_1	
q_proj			Max diff: 0.00011727213859558105
k_proj			Max diff: 6.064772605895996e-05
v_proj			Max diff: 5.958974361419678e-05
o_proj			Max diff: 0.00011539459228515625
gate_proj			Max diff: 0.00011786818504333496
up_proj			Max diff: 0.00011393427848815918
down_proj			Max diff: 0.0004115104675292969


Qwen3DecoderLayer_2	
q_proj			Max diff: 0.0001093745231628418
k_proj			Max diff: 0.00011396408081054688
v_proj			Max diff: 6.021559238433838e-05
o_proj			Max diff: 0.00011178851127624512
gate_proj			Max diff: 0.00011274218559265137
up_proj			Max diff: 0.00012171268463134766
down_proj			Max diff: 0.00024014711380004883


Qwen3DecoderLayer_3	
q_proj			Max dif

In [25]:
import torch.nn as nn
import torch.nn.functional as F


class QuantLinear(nn.Module):
    def __init__(self, qweight, scales, zeros, g_idx, d_in, bias=None):
        super().__init__()
        self.d_in = d_in
        self.register_buffer("qweight", qweight)                                   # [d_out, d_in/2] uint8
        self.register_buffer("scales", torch.stack(scales, dim=1).half())          # [d_out, num_blocks]
        self.register_buffer("zeros", torch.stack(zeros, dim=1).to(torch.uint8))   # [d_out, num_blocks], values 0-15
        self.register_buffer("g_idx", g_idx)                                       # [d_in] -> block number
        self.bias = bias
         
    def dequantize(self):
        q = torch.empty(self.qweight.shape[0], self.qweight.shape[1] * 2,
                        dtype=torch.uint8, device=self.qweight.device)
        q[:, 0::2] = self.qweight & 0x0F
        q[:, 1::2] = (self.qweight >> 4) & 0x0F
        q = q[:, :self.d_in].half()
        scale = self.scales[:, self.g_idx]             # [d_out, d_in], each column gets its block's scale
        zero = self.zeros[:, self.g_idx].half()
        return (q - zero) * scale

    def forward(self, x):
        W = self.dequantize().to(x.dtype)
        return F.linear(x, W, self.bias)

In [ ]:
for layer_idx, layer in enumerate(model.model.layers):
    for name in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]:
        parent = layer.self_attn if hasattr(layer.self_attn, name) else layer.mlp
        old = getattr(parent, name)
        pw = packed_weights[layer_idx][name]
        setattr(parent, name, QuantLinear(
            pw["qweight"].cpu(), pw["scales"], pw["zeros"], pw["g_idx"].cpu(),
            d_in=old.in_features, bias=old.bias,
        ))

In [27]:
sample = calib_batches[0][:1, :128]

with torch.no_grad():
    model.to(device_gpu)
    logits_fake = model(sample.to(device_gpu)).logits.cpu()
    model.to(device_cpu)

    #swap the false qunatized layers with a custom layer that contains real qunatized weights of int4
    for layer_idx, layer in enumerate(model.model.layers):
        for name in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]:
            parent = layer.self_attn if hasattr(layer.self_attn, name) else layer.mlp
            old = getattr(parent, name)
            pw = packed_weights[layer_idx][name]
            setattr(parent, name, QuantLinear(
                pw["qweight"].cpu(), pw["scales"], pw["zeros"], pw["g_idx"].cpu(),
                d_in=old.in_features, bias=old.bias,
            ))

    #call the model
    
    model.to(device_gpu)
    logits_real = model(sample.to(device_gpu)).logits.cpu()
    model.to(device_cpu)
print((logits_fake - logits_real).abs().max())

tensor(0.1094, dtype=torch.float16)


In [28]:
V = logits_fake.shape[-1]
lf, lr = logits_fake.float().reshape(-1, V), logits_real.float().reshape(-1, V)
print("relative err:", ((lf - lr).norm() / lf.norm()).item())
print("top-1 agreement:", (lf.argmax(-1) == lr.argmax(-1)).float().mean().item())

relative err: 0.0019860127940773964
top-1 agreement: 1.0


In [29]:
from datasets import load_dataset

wikitext = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
text = "\n\n".join(wikitext["text"])

enc = tokenizer(text, return_tensors="pt").input_ids

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors


In [36]:
#perplexity
model_orig = AutoModelForCausalLM.from_pretrained( model_name,
                                 dtype = torch.float16,
                                 device_map = 'cpu')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [38]:
device_gpu_2 = torch.device('cuda:1')

In [43]:
@torch.no_grad()
def compute_perplexity(m, enc, seq_len=2048, num_chunks=20):
    m.to(device_gpu_2)
    m.eval()
    nlls = []
    for i in range(num_chunks):
        chunk = enc[:, i*seq_len:(i+1)*seq_len]
        if chunk.shape[1] < seq_len:
            break
        chunk = chunk.to(device_gpu_2)
        out = m(chunk, labels=chunk, use_cache=False)
        nlls.append(out.loss.float() * seq_len)
        del out, chunk
    m.to(device_cpu)
    torch.cuda.empty_cache()
    print(f'Completed Computation for Model 1')
    return torch.exp(torch.stack(nlls).sum() / (len(nlls) * seq_len)).item()

ppl_orig  = compute_perplexity(model_orig, enc)
ppl_quant = compute_perplexity(model, enc)      # swapped model with QuantLinear layers

print(f"original:  {ppl_orig:.3f}")
print(f"int4:      {ppl_quant:.3f}")
print(f"increase:  {(ppl_quant / ppl_orig - 1) * 100:.2f}%")

Completed Computation for Model 1
Completed Computation for Model 1
original:  13.254
int4:      13.671
increase:  3.15%


In [44]:
prompt = "Explain how photosynthesis works in one paragraph."
inputs = tokenizer(prompt, return_tensors="pt").to(device_gpu_2)

model_orig.to(device_gpu_2)
out_orig = model_orig.generate(**inputs, max_new_tokens=100, do_sample=False)
model_orig.to(device_cpu)

model.to(device_gpu_2)
out_quant = model.generate(**inputs, max_new_tokens=100, do_sample=False)
model.to(device_cpu)

print("ORIGINAL:\n", tokenizer.decode(out_orig[0]))
print("\nQUANTIZED:\n", tokenizer.decode(out_quant[0]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ORIGINAL:
 Explain how photosynthesis works in one paragraph. Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy, storing it in the form of glucose. This process occurs in the chloroplasts of plant cells, where chlorophyll, a green pigment, absorbs sunlight. The overall equation for photosynthesis is 6CO₂ + 6H₂O + light → C₆H₁₂O₆ + 6O₂. During photosynthesis, carbon dioxide from the air and water

QUANTIZED:
 Explain how photosynthesis works in one paragraph. Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy from the sun into chemical energy stored in glucose. This process occurs in the chloroplasts of plant cells, where chlorophyll, a green pigment, captures sunlight. The overall equation for photosynthesis is 6CO2 + 6H2O + light → C6H12O6 + 6O2. During the light-dependent reactions, light energy is used to split water molecules into
